# Choosing when to observe a UI

**Goal:** Inspect how a wait budget affects which UI changes arrive later.

**Input:** Six hand-written, synthetic UI transitions held in memory.

**Conclusion:** At the initial 500 ms budget, three of the five UI changes arrive later.
These examples are not benchmark results or model predictions. Adjust the slider to
inspect other budgets.

In [1]:
import sys

if sys.version_info[:2] != (3, 14):
    raise RuntimeError(
        "Python 3.14 is required. Run `uv python install 3.14` and `uv sync`, "
        "then open this notebook with `uv run jupyter lab notebooks/`."
    )

In [2]:
import polars as pl
from IPython.display import Markdown, display
from ipywidgets import IntSlider, interactive_output

In [3]:
transitions = pl.DataFrame(
    {
        "action": [
            "type query",
            "submit search",
            "open menu",
            "open details",
            "navigate back",
            "click disabled button",
        ],
        "before_ui": ["empty form", "filled form", "results", "results", "details", "dialog"],
        "after_ui": ["filled form", "results", "menu", "details", "results", "dialog"],
        "latency_ms": [260, 650, 120, 1100, 900, 50],
    }
).with_columns((pl.col("before_ui") != pl.col("after_ui")).alias("state_changed"))
transitions

action,before_ui,after_ui,latency_ms,state_changed
str,str,str,i64,bool
"""type query""","""empty form""","""filled form""",260,true
"""submit search""","""filled form""","""results""",650,true
"""open menu""","""results""","""menu""",120,true
"""open details""","""results""","""details""",1100,true
"""navigate back""","""details""","""results""",900,true
"""click disabled button""","""dialog""","""dialog""",50,false


## Wait budget

The slider updates the table automatically. The disabled button leaves the UI unchanged
and is excluded from the count.

In [4]:
wait_budget = IntSlider(
    min=0,
    max=1500,
    step=100,
    value=500,
    description="Wait budget (ms)",
    style={"description_width": "initial"},
)


def display_wait_result(budget_ms: int) -> None:
    over_budget = transitions.filter(
        pl.col("state_changed") & (pl.col("latency_ms") > budget_ms)
    ).sort("latency_ms")
    display(
        Markdown(f"**{over_budget.height} toy UI changes** arrive after the {budget_ms} ms budget.")
    )
    display(over_budget)


wait_result = interactive_output(display_wait_result, {"budget_ms": wait_budget})
display(wait_budget, wait_result)

IntSlider(value=500, description='Wait budget (ms)', max=1500, step=100, style=SliderStyle(description_width='…

Output()